<a href="https://colab.research.google.com/github/Pranayshukla0610/Transformers/blob/main/MiniGPT_Transformer_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import sys
import torch

print("Python version:")
print(sys.version)

print("\nPyTorch version:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

PyTorch version:
2.11.0+cpu

CUDA available:
False


In [6]:
device = torch.device('cpu')
print("Using device:",device)

Using device: cpu


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math
import time
import random
import re
import os

from collections import Counter

In [9]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

file_path = "shakespeare.txt"

try:
    urllib.request.urlretrieve(url, file_path)
    print("Dataset downloaded successfully.")

except Exception as e:
    print("Download failed.")
    print("Error:", e)

Dataset downloaded successfully.


In [10]:
with open(file_path, 'r', encoding='utf-8') as f:
  raw_text = f.read()

print("Characters:",len(raw_text))
print("\nFirst 1000 characters:")
print(raw_text[:1000])

Characters: 1115394

First 1000 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hu

In [11]:
def clean_text(text):
  text = text.replace("\r","")

  text = re.sub(r"[ \t]+"," ",text)

  text = re.sub(r"\n{3,}","\n\n",text)

  text = "\n".join(line.strip() for line in text.split("\n"))

  return text.strip()

cleaned_text = clean_text(raw_text)

print("Original characters:", len(raw_text))
print("Cleaned characters:", len(cleaned_text))

print("\nSample:\n")
print(cleaned_text[:1000])

Original characters: 1115394
Cleaned characters: 1115373

Sample:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods 

In [12]:
def normalize_text(text):
  text = text.replace("\t", " ")

  text = text.replace("\r\n", "\n")
  text = text.replace("\r", "\n")

  return text

text = normalize_text(cleaned_text)
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [13]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocabulary Size:",vocab_size)
print("\nCharacters:")
print(chars)

Vocabulary Size: 65

Characters:
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [14]:
#Create Encoder and Decoder

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print("Character to ID:")
print(list(stoi.items())[:20])

print("\nID to Character:")
print(list(itos.items())[:20])

Character to ID:
[('\n', 0), (' ', 1), ('!', 2), ('$', 3), ('&', 4), ("'", 5), (',', 6), ('-', 7), ('.', 8), ('3', 9), (':', 10), (';', 11), ('?', 12), ('A', 13), ('B', 14), ('C', 15), ('D', 16), ('E', 17), ('F', 18), ('G', 19)]

ID to Character:
[(0, '\n'), (1, ' '), (2, '!'), (3, '$'), (4, '&'), (5, "'"), (6, ','), (7, '-'), (8, '.'), (9, '3'), (10, ':'), (11, ';'), (12, '?'), (13, 'A'), (14, 'B'), (15, 'C'), (16, 'D'), (17, 'E'), (18, 'F'), (19, 'G')]


In [15]:
#Encode Function

def encode(text):
  return[stoi[ch] for ch in text]

def decode(ids):
  return ''.join(itos[i] for i in ids)

In [16]:
sample = "Hello"

encoded = encode(sample)
decoded = decode(encoded)

print("Original:")
print(sample)

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(decoded)

Original:
Hello

Encoded:
[20, 43, 50, 50, 53]

Decoded:
Hello


In [17]:
#Create Dataset

data = torch.tensor(encode(text), dtype=torch.long)

print("Total tokens:", len(data))

Total tokens: 1115373


In [18]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens: 1003835
Validation tokens: 111538


In [19]:
#Create Training Batches

batch_size = 16
block_size = 64


def get_batch(split):

    data_source = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data_source) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data_source[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [20]:
X, Y = get_batch("train")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: torch.Size([16, 64])
Y shape: torch.Size([16, 64])


In [21]:
vocab_size = len(chars)

embedding_dim = 64
num_heads = 4
num_layers = 2

dropout = 0.1

print("Vocabulary:", vocab_size)
print("Embedding dimension:", embedding_dim)
print("Attention heads:", num_heads)
print("Transformer layers:", num_layers)

Vocabulary: 65
Embedding dimension: 64
Attention heads: 4
Transformer layers: 2


In [22]:
class Head(nn.Module):

    def __init__(self, head_size):

        super().__init__()

        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)

        # Causal mask
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        B, T, C = x.shape

        # Create Query, Key and Value
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Attention scores
        wei = q @ k.transpose(-2, -1)

        # Scale
        wei = wei / math.sqrt(k.size(-1))

        # Causal mask
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        # Convert to probabilities
        wei = F.softmax(wei, dim=-1)

        # Dropout
        wei = self.dropout(wei)

        # Weighted aggregation
        out = wei @ v

        return out

In [24]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):

        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            num_heads * head_size,
            embedding_dim
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        # Run all attention heads
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        # Project back to embedding dimension
        out = self.projection(out)

        out = self.dropout(out)

        return out

In [25]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(embedding_dim, 4 * embedding_dim),

            nn.ReLU(),

            nn.Linear(4 * embedding_dim, embedding_dim),

            nn.Dropout(dropout)
        )


    def forward(self, x):

        return self.network(x)

In [26]:
class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        head_size = embedding_dim // num_heads

        self.attention = MultiHeadAttention(
            num_heads,
            head_size
        )

        self.feed_forward = FeedForward(
            embedding_dim
        )

        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)


    def forward(self, x):

        # Attention + Residual Connection
        x = x + self.attention(
            self.ln1(x)
        )

        # Feed Forward + Residual Connection
        x = x + self.feed_forward(
            self.ln2(x)
        )

        return x

In [27]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers
    ):

        super().__init__()

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Positional embeddings
        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads
                )
                for _ in range(num_layers)
            ]
        )

        # Final LayerNorm
        self.ln_final = nn.LayerNorm(
            embedding_dim
        )

        # Output layer
        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )


    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        token_emb = self.token_embedding(idx)

        # Position embeddings
        positions = torch.arange(
            T,
            device=device
        )

        pos_emb = self.position_embedding(
            positions
        )

        # Combine token + position information
        x = token_emb + pos_emb

        # Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_final(x)

        # Convert to vocabulary logits
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits_flat = logits.view(
                B * T,
                C
            )

            targets_flat = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits_flat,
                targets_flat
            )

        return logits, loss

In [28]:
model = MiniGPT(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers
).to(device)

print(model)

MiniGPT(
  (token_embedding): Embedding(65, 64)
  (position_embedding): Embedding(64, 64)
  (blocks): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=64, out_features=16, bias=False)
            (query): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (projection): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward): FeedForward(
        (network): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affi

In [29]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 112193
Trainable parameters: 112193


In [30]:
X, Y = get_batch("train")

logits, loss = model(X, Y)

print("Input shape:", X.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([16, 64])
Logits shape: torch.Size([16, 64, 65])
Initial loss: 4.358436107635498


In [31]:
learning_rate = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

In [32]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    losses = {}

    for split in ["train", "val"]:

        split_losses = []

        for _ in range(20):

            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            split_losses.append(
                loss.item()
            )

        losses[split] = sum(split_losses) / len(split_losses)

    model.train()

    return losses

In [33]:
max_iters = 1000

eval_interval = 100

start_time = time.time()

for step in range(max_iters):

    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {step:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f}"
        )

    # Get batch
    X, Y = get_batch("train")

    # Forward pass
    logits, loss = model(X, Y)

    # Clear gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()


training_time = time.time() - start_time

print("\nTraining completed.")
print(f"Training time: {training_time / 60:.2f} minutes")

Step    0 | Train Loss: 4.3371 | Val Loss: 4.3420
Step  100 | Train Loss: 3.1622 | Val Loss: 3.1865
Step  200 | Train Loss: 2.8294 | Val Loss: 2.8455
Step  300 | Train Loss: 2.6842 | Val Loss: 2.7030
Step  400 | Train Loss: 2.6299 | Val Loss: 2.6374
Step  500 | Train Loss: 2.5816 | Val Loss: 2.5946
Step  600 | Train Loss: 2.5311 | Val Loss: 2.5504
Step  700 | Train Loss: 2.5045 | Val Loss: 2.5139
Step  800 | Train Loss: 2.5038 | Val Loss: 2.5013
Step  900 | Train Loss: 2.4919 | Val Loss: 2.4767

Training completed.
Training time: 0.66 minutes


In [34]:
embedding_dim = 64
num_heads = 4
num_layers = 2
batch_size = 16
block_size = 64
max_iters = 1000

In [35]:
final_losses = estimate_loss()

print("Final Training Loss:",
      final_losses["train"])

print("Final Validation Loss:",
      final_losses["val"])

Final Training Loss: 2.464976596832275
Final Validation Loss: 2.4628539204597475


In [36]:
import math

train_perplexity = math.exp(final_losses["train"])
val_perplexity = math.exp(final_losses["val"])

print("Train Perplexity:",
      train_perplexity)

print("Validation Perplexity:",
      val_perplexity)

Train Perplexity: 11.76320685245599
Validation Perplexity: 11.738263853527954


In [37]:
@torch.no_grad()
def generate(idx, max_new_tokens):

    model.eval()

    for _ in range(max_new_tokens):

        # Keep only the last block_size tokens
        idx_cond = idx[:, -block_size:]

        # Get predictions
        logits, _ = model(idx_cond)

        # Take the final time step
        logits = logits[:, -1, :]

        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample next token
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append new token
        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    model.train()

    return idx

In [38]:
prompt = "ROMEO:"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated = generate(
    context,
    max_new_tokens=300
)

result = decode(
    generated[0].tolist()
)

print(result)

ROMEO:
Go hasarere bnof ve heet, he bis p derompo-cike and famest ine!
BQefay:
Th.
ALANong co g she MEviloqud beret,!
I tand'grfrend bus a, I tobrul fayan ads, d ce t kyoiblit hisen.

Havemy, sewelowes brer I othind beexI:
Wid icis, tore w bH as!

Hgrenors !
R'rurteindod thak, woukxelll ler tre lor thero 


In [39]:
@torch.no_grad()
def generate_with_temperature(
    idx,
    max_new_tokens,
    temperature=1.0
):

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    model.train()

    return idx

In [40]:
prompt = "ROMEO:"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

for temperature in [0.5, 1.0, 1.5]:

    generated = generate_with_temperature(
        context,
        max_new_tokens=200,
        temperature=temperature
    )

    print("\n" + "=" * 70)
    print("Temperature:", temperature)
    print("=" * 70)

    print(
        decode(generated[0].tolist())
    )


Temperature: 0.5
ROMEO:
I, therthare the thes thand mar pe thero the me athy lowis theren d go t he withithe theaner s t he the athee m,
I windere hishice the s o oure be the the non ane hare to s atathe
He y tharor t athan

Temperature: 1.0
ROMEO:
Thout, ling s cheet, shics findy Ythailuin tht fon othse sor dem whe hecaleld,
Di! thir meshis t h'n etrthente,
LAner s o tuntate!
EBy
wovey s.
I:
MBETThof rtheorthay d t Paimatu den,


LALKLARIISs w

Temperature: 1.5
ROMEO:
Tntor; th ba'd anore?:
yet litingps ARENDUStis:
K:
PSANelfatholdhentfatnDherd
Thsp:
Te scto'!
J pr, lde afud bus o wo,
Ym
?


Fome-fo.BElicANIWay,,
XSfarr OY d! to mef de e iriceayomy ouhat! chaghion


In [41]:
custom_text = """
ROMEO:
Love is a strange and powerful force.
The heart speaks when the world is silent.

JULIET:
My heart knows what my mind cannot explain.
Love can make the darkest night beautiful.

ROMEO:
If love is a dream, then let me dream forever.
The heart remembers what the mind forgets.
"""

In [42]:
new_chars = sorted(
    list(set(custom_text) - set(chars))
)

print("New characters:", new_chars)

New characters: []


In [43]:
fine_tune_data = torch.tensor(
    encode(custom_text),
    dtype=torch.long
)

print("Fine-tuning tokens:",
      len(fine_tune_data))

Fine-tuning tokens: 283


In [44]:
def get_finetune_batch():

    ix = torch.randint(
        len(fine_tune_data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        fine_tune_data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        fine_tune_data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [45]:
fine_tune_lr = 1e-4

fine_tune_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=fine_tune_lr
)

In [46]:
fine_tune_steps = 150

for step in range(fine_tune_steps):

    X, Y = get_finetune_batch()

    logits, loss = model(X, Y)

    fine_tune_optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    fine_tune_optimizer.step()

    if step % 25 == 0:
        print(
            f"Fine-tuning Step {step} | "
            f"Loss: {loss.item():.4f}"
        )

Fine-tuning Step 0 | Loss: 2.4011
Fine-tuning Step 25 | Loss: 2.2305
Fine-tuning Step 50 | Loss: 2.1213
Fine-tuning Step 75 | Loss: 2.0574
Fine-tuning Step 100 | Loss: 1.9799
Fine-tuning Step 125 | Loss: 1.8887


In [47]:
prompt = "ROMEO:"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long
)

generated = generate_with_temperature(
    context,
    max_new_tokens=300,
    temperature=0.8
)

print(
    decode(generated[0].tolist())
)

ROMEO:
IO:
CTow t t amis t d:
Theart
GIforun thkeakn, be d rthetharearerl, m s thererenghevenile t heaVdrl
G.
YOMEMTopeat e s co ast amanornd n thare make the mearI mave.
Men t t arest knkesethern.
CDhe owheaks ndrtzeake anore lon hen at'st lere
IOThes ome m llearlkis whe whe ighea anonthed lenof m mares 


In [48]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "vocab": chars,
    "embedding_dim": embedding_dim,
    "num_heads": num_heads,
    "num_layers": num_layers,
    "block_size": block_size
}

torch.save(
    checkpoint,
    "minigpt_shakespeare.pt"
)

print("Model saved successfully.")

Model saved successfully.


In [49]:
import os

print(
    "File size:",
    os.path.getsize(
        "minigpt_shakespeare.pt"
    ) / (1024 * 1024),
    "MB"
)

File size: 1.4718999862670898 MB


In [50]:
checkpoint = torch.load(
    "minigpt_shakespeare.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [51]:
def generate_text(
    prompt,
    max_new_tokens=200,
    temperature=0.8
):

    if not prompt:
        prompt = "ROMEO:"

    context = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    start = time.time()

    generated = generate_with_temperature(
        context,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )

    end = time.time()

    text = decode(
        generated[0].tolist()
    )

    latency = end - start

    return text, latency

In [52]:
text, latency = generate_text(
    prompt="ROMEO:",
    max_new_tokens=200,
    temperature=0.8
)

print(text)

print("\nGeneration latency:",
      round(latency, 3),
      "seconds")

ROMEO:
T joie, beam, my myethe n it wis fkn core my fulerowhert theat The fond heame fothe tham, whe.
Be foupert d, landarlat orthart iamy akn s he marerd n toureveamean
Heat nen bere ar weaneand thet menke

Generation latency: 0.73 seconds


In [54]:
monitoring_log = []


def monitored_generate(
    prompt,
    max_new_tokens=200,
    temperature=0.8
):

    start = time.time()

    text, latency = generate_text(
        prompt,
        max_new_tokens,
        temperature
    )

    end = time.time()

    record = {
        "prompt": prompt,
        "latency_seconds": round(
            latency,
            3
        ),
        "output_length": len(text),
        "temperature": temperature,
        "timestamp": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    monitoring_log.append(record)

    return text

In [55]:
print(
    monitored_generate(
        "ROMEO:",
        max_new_tokens=100
    )
)

print("\n" + "-" * 60)

print(
    monitored_generate(
        "JULIET:",
        max_new_tokens=100
    )
)

print("\n" + "-" * 60)

print(
    monitored_generate(
        "KING:",
        max_new_tokens=100
    )
)

ROMEO:
RELows keat f keala cartin anort y dardrin wcen heavean theare frealeve.
Lonowe, e am fove ghen iwh

------------------------------------------------------------
JULIET:
I d wheail ss t foveanowa wret in fovea the y ce In drthexdy forethe
Thelis, is heanid s d stheaghe

------------------------------------------------------------
KING:
MERBEF:
CO:
Rnd t foushake ce are t an theas is the im ainceam akene s d m.
RTheupeat s orearea lat


In [56]:
def quality_check(text):

    issues = []

    if len(text.strip()) < 20:
        issues.append(
            "Output is too short"
        )

    if text.count("\n") > 50:
        issues.append(
            "Too many line breaks"
        )

    if not text.strip():
        issues.append(
            "Empty output"
        )

    if len(issues) == 0:
        return "PASS"

    return " | ".join(issues)

In [57]:
generated_text = monitored_generate(
    "ROMEO:",
    max_new_tokens=150
)

print(generated_text)

print(
    "\nQuality:",
    quality_check(generated_text)
)

ROMEO:
MO:
In mpearpen d orka fond me sond hthe ist hereat he dor.

Wo d s lakeart whavelfre han ghe ben:
The l powe heanthert s s lest t are the nearare in

Quality: PASS


In [58]:
def ask_model(prompt):

    text, latency = generate_text(
        prompt=prompt,
        max_new_tokens=200,
        temperature=0.8
    )

    print("Generated Text")
    print("=" * 60)
    print(text)

    print("\nLatency:",
          round(latency, 3),
          "seconds")

In [59]:
ask_model("ROMEO:")

Generated Text
ROMEO:
Theart thar in s mesthere dulet enowhe meeakn alen mean 
Marene benoroveren Joufule imerthe frke heancale t winoveave im m.
Thare,

HIX
RThe wheais t my st nt marerurerlarevAnn sthavend keen he'spear

Latency: 1.332 seconds


In [60]:
ask_model("JULIET:")

Generated Text
JULIET:
ARIULI y fo re nove lo nonowhe mpand d wis, marend whe ke ken l.

henisherDUThe trears s:
Me hen weareant heareemy fowhe re t he, t he mat lendlend he them.
GULIOLolINMELJex drebereat eaverevet pe t 

Latency: 0.573 seconds
